# Reproduce additional-analysis figures

This notebook regenerates the additional robustness figures directly from the archived machine-readable result tables in this repository. It performs plotting only; no model fitting or feature selection is rerun.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Locate the repository root from the current working directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "NIBFS_REPRODUCIBILITY_PACKAGE.marker").is_file()), None)
if ROOT is None:
    raise RuntimeError("Repository root not found. Run this notebook from within the NIBFS repository.")
BASE = ROOT / "additional_robustness_analyses" / "results_additional"
FIGDIR = ROOT / "additional_robustness_analyses" / "publication_figures"
FIGDIR.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)


In [ ]:
strict_repeat = pd.read_csv(BASE / "02_repeated_fold_fitted_5x5" / "repeated_fold_fitted_5x5_stability_by_repeat.csv")
ss_repeat = pd.read_csv(BASE / "03_stability_selection_repeated_10x5" / "all_methods_plus_stability_selection_stability_by_repeat.csv")
ss_summary = pd.read_csv(BASE / "03_stability_selection_repeated_10x5" / "all_methods_plus_stability_selection_stability_summary.csv")
auc = pd.read_csv(BASE / "03_stability_selection_repeated_10x5" / "reported_existing_plus_stability_selection_lr_auc.csv")
boot = pd.read_csv(BASE / "01_gse15852_paired" / "GSE15852_pair_cluster_bootstrap_distribution.csv.gz")
gse_auc = pd.read_csv(BASE / "01_gse15852_paired" / "GSE15852_pair_cluster_bootstrap_auc.csv")


In [ ]:
# Supplementary Figure S5: repeated strict training-fold-fitted stability.
plt.figure(figsize=(8.2, 5.6))
for method, g in strict_repeat.groupby("Method", sort=False):
    g = g.sort_values("Repeat")
    plt.plot(g["Repeat"], g["Mean_Jaccard"], marker="o", linewidth=1.8, label=method)
plt.xticks(sorted(strict_repeat["Repeat"].unique()))
plt.ylim(0, 1.0)
plt.xlabel("Repeat")
plt.ylabel("Mean within-repeat pairwise Jaccard")
plt.title("Repeated strict training-fold-fitted stability (5×5-fold)")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIGDIR / "Figure_S5_Strict_FoldFitted_5x5_Jaccard.png", dpi=300, bbox_inches="tight")
plt.savefig(FIGDIR / "Figure_S5_Strict_FoldFitted_5x5_Jaccard.pdf", bbox_inches="tight")
strict_repeat.to_csv(FIGDIR / "Figure_S5_source_data.csv", index=False)
plt.show(); plt.close()


In [ ]:
# Supplementary Figure S6: stability-selection comparator.
plt.figure(figsize=(8.6, 5.8))
for method, g in ss_repeat.groupby("Method", sort=False):
    g = g.sort_values("Repeat")
    plt.plot(g["Repeat"], g["Mean_Jaccard"], marker="o", linewidth=1.6, label=method)
plt.xticks(sorted(ss_repeat["Repeat"].unique()))
plt.ylim(0, 1.0)
plt.xlabel("Repeat")
plt.ylabel("Mean within-repeat pairwise Jaccard")
plt.title("Repeated 10×5-fold stability with stability-selection comparator")
plt.legend(frameon=False, ncol=2)
plt.tight_layout()
plt.savefig(FIGDIR / "Figure_S6_StabilitySelection_Comparator_10x5.png", dpi=300, bbox_inches="tight")
plt.savefig(FIGDIR / "Figure_S6_StabilitySelection_Comparator_10x5.pdf", bbox_inches="tight")
ss_repeat.to_csv(FIGDIR / "Figure_S6_source_data.csv", index=False)
plt.show(); plt.close()


In [ ]:
# Supplementary Figure S8: stability-discrimination relationship.
trade = ss_summary[["Method", "Mean_Jaccard"]].merge(auc[["Method", "LR_OOF_ROC_AUC_mean"]], on="Method", how="inner")
plt.figure(figsize=(7.6, 5.8))
plt.scatter(trade["Mean_Jaccard"], trade["LR_OOF_ROC_AUC_mean"], s=70)
for _, r in trade.iterrows():
    plt.annotate(r["Method"], (r["Mean_Jaccard"], r["LR_OOF_ROC_AUC_mean"]), xytext=(6, 5), textcoords="offset points", fontsize=9)
plt.xlabel("Repeated mean Jaccard stability")
plt.ylabel("LR out-of-fold ROC-AUC")
plt.title("Stability-discrimination relationship under repeated 10×5-fold evaluation")
plt.tight_layout()
plt.savefig(FIGDIR / "Figure_S8_Stability_vs_LR_AUC.png", dpi=300, bbox_inches="tight")
plt.savefig(FIGDIR / "Figure_S8_Stability_vs_LR_AUC.pdf", bbox_inches="tight")
trade.to_csv(FIGDIR / "Figure_S8_Stability_vs_LR_AUC_source_data.csv", index=False)
plt.show(); plt.close()


In [ ]:
# Supplementary Figure S7: GSE15852 patient-pair bootstrap.
order = ["LR", "RF", "LightGBM"]
data = [boot.loc[boot["Classifier"] == m, "ROC_AUC"].dropna().values for m in order]
plt.figure(figsize=(7.4, 5.6))
plt.boxplot(data, tick_labels=order, showfliers=False)
for i, m in enumerate(order, start=1):
    point = float(gse_auc.loc[gse_auc["Classifier"] == m, "ROC_AUC"].iloc[0])
    plt.scatter(i, point, marker="D", s=45, label="Observed ROC-AUC" if i == 1 else None)
plt.ylim(0.6, 1.0)
plt.xlabel("Classifier")
plt.ylabel("ROC-AUC")
plt.title("GSE15852: 2,000 patient-pair cluster bootstrap replicates")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIGDIR / "Figure_S7_GSE15852_PairedBootstrap.png", dpi=300, bbox_inches="tight")
plt.savefig(FIGDIR / "Figure_S7_GSE15852_PairedBootstrap.pdf", bbox_inches="tight")
gse_auc.to_csv(FIGDIR / "Figure_S7_point_estimates_and_CI.csv", index=False)
plt.show(); plt.close()


Generated figure files and their machine-readable source tables are saved in `additional_robustness_analyses/publication_figures/`.
